# ⚡ حل مشروع السلاسل الزمنية باستهلاك الطاقة (Energy Forecasting Solution)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error

df = pd.read_csv('datasets/energy_consumption_data.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').asfreq('h') # set hourly frequency
print(df.head())

### 📊 الجزء الأول: تفكيك واختبار استقرار السلسلة

In [ ]:
decomp = seasonal_decompose(df['consumption_kwh'], period=24)
fig = decomp.plot()
fig.set_size_inches(10, 8)
plt.savefig('solutions/energy_seasonal_decompose.png')
plt.show()

# ADF Test
result = adfuller(df['consumption_kwh'])
print(f"ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.6f}")
if result[1] < 0.05:
    print("Series is Stationary (No differencing needed)")
else:
    print("Series is Non-Stationary (Needs differencing)")

### 🤖  الجزء الثاني: نماذج التنبؤ (ARIMA & Exponential Smoothing)

In [ ]:
train_size = int(len(df) * 0.9)
train, test = df['consumption_kwh'].iloc[:train_size], df['consumption_kwh'].iloc[train_size:]

# Fit HW Exponential Smoothing
model_es = ExponentialSmoothing(train, seasonal_periods=24, trend='add', seasonal='add')
model_es_fit = model_es.fit()
preds_es = model_es_fit.forecast(len(test))

rmse_es = np.sqrt(mean_squared_error(test, preds_es))
print(f"Holt-Winters Exponential Smoothing RMSE: {rmse_es:.2f} kWh")

plt.figure(figsize=(12, 6))
plt.plot(train.index[-100:], train.iloc[-100:], label='Train')
plt.plot(test.index, test, label='Actual Test')
plt.plot(test.index, preds_es, label='HW Forecast')
plt.title('Energy Consumption Forecasting')
plt.legend()
plt.tight_layout()
plt.savefig('solutions/energy_forecasting_result.png')
plt.show()